In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')

In [2]:
# load the messy data
df = pd.read_csv('marketing_campaign_data_messy.csv')

### Basic Ques about Data

In [5]:
# How does the data looks like?
df.head()

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


In [6]:
# How big is the data?
df.shape

(2020, 12)

In [5]:
# What is the data type of columns?
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   object 
 1   Campaign_Name  2020 non-null   object 
 2   Start_Date     2020 non-null   object 
 3   End_Date       2020 non-null   object 
 4   Channel        1919 non-null   object 
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   object 
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   object 
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   object 
dtypes: float64(2), int64(2), object(8)
memory usage: 189.5+ KB


In [6]:
# Are there any missing values?
df.isna().sum()

 Campaign_ID        0
Campaign_Name       0
Start_Date          0
End_Date            0
Channel           101
Impressions         0
Clicks              0
Spend               0
Conversions       200
Active              0
Clicks           1980
Campaign_Tag        0
dtype: int64

In [7]:
# Are there duplicate values?
df.duplicated().sum()

np.int64(19)

### STEP 1: Header Cleaning

In [8]:
# print(df.columns)
print(df.columns.tolist()) # many columns heading has some space

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("Fixed the columns problems...")
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
Fixed the columns problems...
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


### STEP 2: Type Conversion and Currency Cleaning

In [9]:
# Remove the '$' sign
dirty_spend_mask = df['spend'] = df['spend'].astype(str).str.replace('$', '', regex=False).str.strip()
dirty_spend_mask

# Convert to numeric value(float) for calculation
df['spend'] = pd.to_numeric(df['spend'])

In [10]:
df.head()

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


### STEP 3: Categorical Typos (fuzzy logic)

In [10]:
# fixed this problem
print(df['channel'].unique())

clean_channel = {
    'Facebok': 'Facebook',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads',
    'N/A': np.nan # handling the ghost value
}

df['channel'] = df['channel'].replace(clean_channel)
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


### STEP 4: Handling Mixed Booleans

In [12]:
df.sample(5)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
571,CMP-00572,Q2_Launch_CMP-00572,2023-11-05 00:00:00,2023-11-29,Instagram,12338,389,332.87,74.0,True,NaN,IN
589,CMP-00590,Q1_Launch_CMP-00590,2023-04-06 00:00:00,2023-05-05,Facebook,35911,557,994.68,57.0,No,NaN,FA
882,CMP-00883,Q3_Launch_CMP-00883,2023-04-13 00:00:00,2023-05-01,Google Ads,93705,3558,6985.10,652.0,1,NaN,GO
1228,CMP-01229,Q1_Winter_CMP-01229,19/12/2023,2023-12-30,Google Ads,40224,1800,3291.97,239.0,False,NaN,GO
1728,CMP-01729,Q2_BlackFriday_CMP-01729,2023-04-08 00:00:00,2023-04-22,Google Ads,27292,922,1498.19,109.0,Y,NaN,GO


In [11]:
print(df['active'].unique())

clean_active = {
    'Y': True,
    'Yes': True,
    '1': True,
    1: True,
    'NO': False,
    '0': False,
    0: False
}

df['active'] = df['active'].map(clean_active).fillna(False).astype(bool)
print(df['active'].unique())

['Y' '0' 'No' 'True' 'Yes' '1' 'False']
[ True False]


### STEP 5: Date Parsing

In [17]:
df.sample(5)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
801,CMP-00802,Q2_Summer_CMP-00802,2023-01-09 00:00:00,2023-01-12,Email,19384,702,1147.42,105.0,True,NaN,EM
837,CMP-00838,Q4_Launch_CMP-00838,2023-02-08 00:00:00,2023-02-24,Facebook,43944,885,742.15,103.0,True,NaN,FA
1173,CMP-01174,Q3_Winter_CMP-01174,2023-08-31 00:00:00,2023-09-21,Facebook,40666,1368,1059.01,270.0,True,NaN,FA
915,CMP-00916,Q1_Launch_CMP-00916,2023-12-03,2023-03-14,Instagram,17907,744,714.88,93.0,True,NaN,IN
986,CMP-00987,Q2_Summer_CMP-00987,2023-08-08 00:00:00,2023-09-06,Facebook,24275,680,672.20,68.0,True,NaN,FA


In [ ]:
print(df['start_date'].dtype) # object
print(df['end_date'].dtype) # object

# convert object to date-time
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

print(df['start_date'].dtype)


object
object
datetime64[ns]


### STEP 6: Logical Integrity (clicks and impressions)

In [13]:
df.head()

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,NaN,FA


In [14]:
# here are 2 clicks cols, remove duplicate
df = df.loc[:, ~df.columns.duplicated()] # Keep only the first occurrence of each column name
df.head()

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA


In [15]:
impossible_mask = df['clicks'] > df['impressions']  # it's can not possible
print(df.loc[impossible_mask, ['campaign_id', 'impressions', 'clicks']].head())

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


### STEP 7: Logical Integrity (time travel)

In [16]:
time_travel_mask = df['start_date'] > df['end_date']  # it's can not possible
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head())

    campaign_id start_date    end_date
23    CMP-00024 2023-05-06  2023-05-01
54    CMP-00055 2023-09-01  2023-08-27
71    CMP-00072 2023-02-01  2023-01-27
156   CMP-00157 2023-12-06  2023-12-01
200   CMP-00201 2023-01-11  2023-01-06


### STEP 8: Handling Outliers

In [19]:
# 1. Calculate IQR
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)
IQR = Q3 - Q1

# 2. Define Bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Cap the outliers (Clip them to the upper/lower bounds)
# df['spend_cleaned'] = df['spend'].clip(lower=lower_bound, upper=upper_bound)

print(f"Upper Bound: {upper_bound}")
print(f"Lower Bound: {lower_bound}")

# Keep only the rows that fall within the bounds
df_no_outliers = df[(df['spend'] >= lower_bound) & (df['spend'] <= upper_bound)]
df_no_outliers.count()

Upper Bound: 5620.959999999999
Lower Bound: -2332.5799999999995


campaign_id      1944
campaign_name    1944
start_date       1624
end_date         1944
channel          1846
impressions      1944
clicks           1944
spend            1944
conversions      1754
active           1944
campaign_tag     1944
season           1944
dtype: int64

### STEP 9: Feature Extraction

In [35]:
df.sample(5)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
1963,CMP-01964,Q2_Winter_CMP-01964,2023-06-13,2023-06-27,Facebook,27289,796,1312.45,105.0,False,FA
424,CMP-00425,Q2_Launch_CMP-00425,2023-05-28,2023-06-04,Facebook,97825,4242,4022.63,354.0,True,FA
585,CMP-00586,Q1_Winter_CMP-00586,2023-07-09,2023-07-23,Email,18955,481,743.17,33.0,True,EM
578,CMP-00579,Q2_BlackFriday_CMP-00579,2023-02-08,2023-02-21,TikTok,45425,1832,1670.01,141.0,True,TI
357,CMP-00358,Q4_Launch_CMP-00358,2023-03-26,2023-04-05,Google Ads,37212,839,1470.43,164.0,False,GO


In [18]:
# Extract the middle part
# We split by '_' and take the index 1 (the second item)
df['season'] = df['campaign_name'].str.split('_').str[1]
df.head()

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
